In [1]:
!pip install gymnasium

In [2]:
import gymnasium as gym
import numpy
import random

# Stratégies d'apprentissage

In [3]:
# Politique souple comme politique de comportement
def epsilon_greedy(Q, state, epsilon, n_actions):
    if random.random() < epsilon:
        return random.randint(0, n_actions - 1)
    else:
        return numpy.argmax(Q[state])

In [4]:
def q_learning(environment, gamma, learning_rate, max_episodes):

    n_states = environment.observation_space.n  
    n_actions = environment.action_space.n

    Q = numpy.random.random((n_states, n_actions))

    for episode in range(1, max_episodes + 1):

        epsilon = max(0.1, 0.99 ** (episode - 1))
        lr = max(0.1, learning_rate * (0.99 ** (episode - 1)))

        state, _ = environment.reset()

        total_reward = 0
        done = False

        while not done:
            action = epsilon_greedy(Q, state, epsilon, n_actions)

            next_state, reward, terminated, truncated, _ = environment.step(action)
            done = terminated or truncated

            q_max = numpy.max(Q[next_state])

            Q[state, action] = Q[state, action] + learning_rate * ( # learning_rate peut etre remplacé par lr 
                reward + gamma * q_max - Q[state, action]
            )

            total_reward += reward
            state = next_state

        if episode % 10 == 0: # Pour visualiser l'évolution de epsilon et des récompenses au cours des épisodes
            print(f"Episode {episode} | epsilon = {epsilon:.3f} | reward = {total_reward}")

    return Q

In [5]:
# Stratégie temporaire
def sarsa(environment, gamma, learning_rate, max_episodes):

    n_states = environment.observation_space.n
    n_actions = environment.action_space.n

    Q = numpy.random.random((n_states, n_actions))

    for episode in range(1, max_episodes + 1):

        epsilon = max(0.1, 0.99 ** (episode - 1))
        lr = max(0.1, learning_rate * (0.99 ** (episode - 1)))

        state, _ = environment.reset()

        action = epsilon_greedy(Q, state, epsilon, n_actions)

        total_reward = 0
        done = False

        while not done:

            next_state, reward, terminated, truncated, _ = environment.step(action)
            done = terminated or truncated

            next_action = epsilon_greedy(Q, next_state, epsilon, n_actions)

            Q[state, action] = Q[state, action] + lr * (
                reward + gamma * Q[next_state, next_action] - Q[state, action]
            )

            total_reward += reward

            state = next_state
            action = next_action

        if episode % 10 == 0: # Pour visualiser l'évolution de epsilon et des récompenses au cours des épisodes
            print(f"Episode {episode} | epsilon = {epsilon:.3f} | reward = {total_reward}")

    return Q

# Visualisation d'une politique basée sur les valeurs d'actions

In [6]:
# Visualisation d'une politique greedy 
def display_tabular(q_values, desc, max_steps=100):

    environment = gym.make("FrozenLake-v1", desc=desc, is_slippery=True, render_mode="human")

    state, _ = environment.reset()

    terminated, truncated = False, False
    time_step = 0

    while not (terminated or truncated) and time_step < max_steps:
        environment.render()

        action = numpy.argmax(q_values[state])
        print("Chosen action:", action)

        next_state, _, terminated, truncated, _ = environment.step(action)

        state = next_state
        time_step += 1

    environment.close()

# Personnalisation de l'environnement FrozenLake-v1

In [7]:
class CustomFrozenLake(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.nrow = env.unwrapped.nrow
        self.ncol = env.unwrapped.ncol
        self.desc = env.unwrapped.desc
        self.visited_states = set()

    def reset(self, **kwargs):
        state, info = self.env.reset(**kwargs)
        self.visited_states = set()
        self.visited_states.add(state) 
        return state, info

    def step(self, action):
        current_state = self.env.unwrapped.s

        next_state, _, terminated, truncated, info = self.env.step(action)

        row, col = divmod(next_state, self.ncol)
        tile = self.desc[row][col].decode("utf-8")

        if tile == 'H': # H correspond à un trou
            reward = -5

        elif tile == 'G': # G correspond à un cadeau
            reward = 5

        elif next_state == current_state: # ceci traite le cas où l'agent sort de la grille
            reward = -3

        #elif next_state in self.visited_states: # ceci traite le cas où l'agent revient à des états qui a déja visité
         #   reward = -2   t

        else:
            reward = 0

        self.visited_states.add(next_state)

        return next_state, reward, terminated, truncated, info

# Environnement simple

In [8]:
safe_map = [
    "SFFH",
    "FFFH",
    "FFFH",
    "FFGH"
]

safe_env = gym.make("FrozenLake-v1", desc=safe_map, is_slippery=True)
safe_env = CustomFrozenLake(safe_env)

### a- Q-learning

In [9]:
# Il faut faire une recherche en grille ou aléatoire pour trouver les bons hyperparams ?
gamma = 0.99 
learning_rate = 1
max_episodes = 20000

qValues_qLearning_safeMap = q_learning(safe_env, gamma, learning_rate, max_episodes)
for row in qValues_qLearning_safeMap: # Pour afficher les valeurs des actions dans une format lisible
    print(["{:.2f}".format(x) for x in row])

Episode 10 | epsilon = 0.914 | reward = -29
Episode 20 | epsilon = 0.826 | reward = -96
Episode 30 | epsilon = 0.747 | reward = -38
Episode 40 | epsilon = 0.676 | reward = -5
Episode 50 | epsilon = 0.611 | reward = -10
Episode 60 | epsilon = 0.553 | reward = -26
Episode 70 | epsilon = 0.500 | reward = -20
Episode 80 | epsilon = 0.452 | reward = -17
Episode 90 | epsilon = 0.409 | reward = -11
Episode 100 | epsilon = 0.370 | reward = -17
Episode 110 | epsilon = 0.334 | reward = -35
Episode 120 | epsilon = 0.302 | reward = -8
Episode 130 | epsilon = 0.273 | reward = -28
Episode 140 | epsilon = 0.247 | reward = -1
Episode 150 | epsilon = 0.224 | reward = 5
Episode 160 | epsilon = 0.202 | reward = -1
Episode 170 | epsilon = 0.183 | reward = -49
Episode 180 | epsilon = 0.165 | reward = 5
Episode 190 | epsilon = 0.150 | reward = -11
Episode 200 | epsilon = 0.135 | reward = -14
Episode 210 | epsilon = 0.122 | reward = -20
Episode 220 | epsilon = 0.111 | reward = -1
Episode 230 | epsilon = 0.10

Episode 1900 | epsilon = 0.100 | reward = -1
Episode 1910 | epsilon = 0.100 | reward = -8
Episode 1920 | epsilon = 0.100 | reward = -1
Episode 1930 | epsilon = 0.100 | reward = -17
Episode 1940 | epsilon = 0.100 | reward = -8
Episode 1950 | epsilon = 0.100 | reward = -5
Episode 1960 | epsilon = 0.100 | reward = 5
Episode 1970 | epsilon = 0.100 | reward = 2
Episode 1980 | epsilon = 0.100 | reward = 5
Episode 1990 | epsilon = 0.100 | reward = -10
Episode 2000 | epsilon = 0.100 | reward = -19
Episode 2010 | epsilon = 0.100 | reward = -25
Episode 2020 | epsilon = 0.100 | reward = -22
Episode 2030 | epsilon = 0.100 | reward = -5
Episode 2040 | epsilon = 0.100 | reward = -17
Episode 2050 | epsilon = 0.100 | reward = -8
Episode 2060 | epsilon = 0.100 | reward = -25
Episode 2070 | epsilon = 0.100 | reward = -8
Episode 2080 | epsilon = 0.100 | reward = -19
Episode 2090 | epsilon = 0.100 | reward = -5
Episode 2100 | epsilon = 0.100 | reward = -7
Episode 2110 | epsilon = 0.100 | reward = -34
Epis

Episode 3720 | epsilon = 0.100 | reward = 5
Episode 3730 | epsilon = 0.100 | reward = -22
Episode 3740 | epsilon = 0.100 | reward = -25
Episode 3750 | epsilon = 0.100 | reward = -19
Episode 3760 | epsilon = 0.100 | reward = -96
Episode 3770 | epsilon = 0.100 | reward = -7
Episode 3780 | epsilon = 0.100 | reward = -13
Episode 3790 | epsilon = 0.100 | reward = -5
Episode 3800 | epsilon = 0.100 | reward = -47
Episode 3810 | epsilon = 0.100 | reward = -5
Episode 3820 | epsilon = 0.100 | reward = -4
Episode 3830 | epsilon = 0.100 | reward = -13
Episode 3840 | epsilon = 0.100 | reward = 5
Episode 3850 | epsilon = 0.100 | reward = -11
Episode 3860 | epsilon = 0.100 | reward = -31
Episode 3870 | epsilon = 0.100 | reward = -5
Episode 3880 | epsilon = 0.100 | reward = -17
Episode 3890 | epsilon = 0.100 | reward = -45
Episode 3900 | epsilon = 0.100 | reward = -10
Episode 3910 | epsilon = 0.100 | reward = -20
Episode 3920 | epsilon = 0.100 | reward = -11
Episode 3930 | epsilon = 0.100 | reward = -

Episode 5600 | epsilon = 0.100 | reward = -13
Episode 5610 | epsilon = 0.100 | reward = 5
Episode 5620 | epsilon = 0.100 | reward = -22
Episode 5630 | epsilon = 0.100 | reward = -4
Episode 5640 | epsilon = 0.100 | reward = -28
Episode 5650 | epsilon = 0.100 | reward = -16
Episode 5660 | epsilon = 0.100 | reward = 2
Episode 5670 | epsilon = 0.100 | reward = 5
Episode 5680 | epsilon = 0.100 | reward = -14
Episode 5690 | epsilon = 0.100 | reward = 2
Episode 5700 | epsilon = 0.100 | reward = -17
Episode 5710 | epsilon = 0.100 | reward = 2
Episode 5720 | epsilon = 0.100 | reward = -14
Episode 5730 | epsilon = 0.100 | reward = -1
Episode 5740 | epsilon = 0.100 | reward = -41
Episode 5750 | epsilon = 0.100 | reward = -17
Episode 5760 | epsilon = 0.100 | reward = -1
Episode 5770 | epsilon = 0.100 | reward = -13
Episode 5780 | epsilon = 0.100 | reward = -29
Episode 5790 | epsilon = 0.100 | reward = -20
Episode 5800 | epsilon = 0.100 | reward = -5
Episode 5810 | epsilon = 0.100 | reward = -5
Epi

Episode 7500 | epsilon = 0.100 | reward = 5
Episode 7510 | epsilon = 0.100 | reward = -14
Episode 7520 | epsilon = 0.100 | reward = 2
Episode 7530 | epsilon = 0.100 | reward = -8
Episode 7540 | epsilon = 0.100 | reward = -8
Episode 7550 | epsilon = 0.100 | reward = -8
Episode 7560 | epsilon = 0.100 | reward = -14
Episode 7570 | epsilon = 0.100 | reward = -35
Episode 7580 | epsilon = 0.100 | reward = -7
Episode 7590 | epsilon = 0.100 | reward = 5
Episode 7600 | epsilon = 0.100 | reward = 5
Episode 7610 | epsilon = 0.100 | reward = -17
Episode 7620 | epsilon = 0.100 | reward = 5
Episode 7630 | epsilon = 0.100 | reward = -1
Episode 7640 | epsilon = 0.100 | reward = -1
Episode 7650 | epsilon = 0.100 | reward = -34
Episode 7660 | epsilon = 0.100 | reward = -56
Episode 7670 | epsilon = 0.100 | reward = -5
Episode 7680 | epsilon = 0.100 | reward = -26
Episode 7690 | epsilon = 0.100 | reward = 5
Episode 7700 | epsilon = 0.100 | reward = -10
Episode 7710 | epsilon = 0.100 | reward = -7
Episode 

...

Episode 11270 | epsilon = 0.100 | reward = 5
Episode 11280 | epsilon = 0.100 | reward = -13
Episode 11290 | epsilon = 0.100 | reward = -10
Episode 11300 | epsilon = 0.100 | reward = -14
Episode 11310 | epsilon = 0.100 | reward = -7
Episode 11320 | epsilon = 0.100 | reward = -4
Episode 11330 | epsilon = 0.100 | reward = 5
Episode 11340 | epsilon = 0.100 | reward = 5
Episode 11350 | epsilon = 0.100 | reward = -19
Episode 11360 | epsilon = 0.100 | reward = -23
Episode 11370 | epsilon = 0.100 | reward = -11
Episode 11380 | epsilon = 0.100 | reward = -38
Episode 11390 | epsilon = 0.100 | reward = -13
Episode 11400 | epsilon = 0.100 | reward = 5
Episode 11410 | epsilon = 0.100 | reward = -7
Episode 11420 | epsilon = 0.100 | reward = -5
Episode 11430 | epsilon = 0.100 | reward = -34
Episode 11440 | epsilon = 0.100 | reward = 2
Episode 11450 | epsilon = 0.100 | reward = -8
Episode 11460 | epsilon = 0.100 | reward = -17
Episode 11470 | epsilon = 0.100 | reward = -7
Episode 11480 | epsilon = 0.1

Episode 13150 | epsilon = 0.100 | reward = -1
Episode 13160 | epsilon = 0.100 | reward = -1
Episode 13170 | epsilon = 0.100 | reward = -16
Episode 13180 | epsilon = 0.100 | reward = -14
Episode 13190 | epsilon = 0.100 | reward = -10
Episode 13200 | epsilon = 0.100 | reward = -70
Episode 13210 | epsilon = 0.100 | reward = -8
Episode 13220 | epsilon = 0.100 | reward = -10
Episode 13230 | epsilon = 0.100 | reward = -44
Episode 13240 | epsilon = 0.100 | reward = -14
Episode 13250 | epsilon = 0.100 | reward = -23
Episode 13260 | epsilon = 0.100 | reward = 2
Episode 13270 | epsilon = 0.100 | reward = -1
Episode 13280 | epsilon = 0.100 | reward = -23
Episode 13290 | epsilon = 0.100 | reward = -19
Episode 13300 | epsilon = 0.100 | reward = -23
Episode 13310 | epsilon = 0.100 | reward = -5
Episode 13320 | epsilon = 0.100 | reward = -62
Episode 13330 | epsilon = 0.100 | reward = -7
Episode 13340 | epsilon = 0.100 | reward = -4
Episode 13350 | epsilon = 0.100 | reward = -26
Episode 13360 | epsilo

Episode 15060 | epsilon = 0.100 | reward = 5
Episode 15070 | epsilon = 0.100 | reward = -1
Episode 15080 | epsilon = 0.100 | reward = -1
Episode 15090 | epsilon = 0.100 | reward = -11
Episode 15100 | epsilon = 0.100 | reward = -10
Episode 15110 | epsilon = 0.100 | reward = -17
Episode 15120 | epsilon = 0.100 | reward = -1
Episode 15130 | epsilon = 0.100 | reward = -8
Episode 15140 | epsilon = 0.100 | reward = -29
Episode 15150 | epsilon = 0.100 | reward = -10
Episode 15160 | epsilon = 0.100 | reward = -5
Episode 15170 | epsilon = 0.100 | reward = -11
Episode 15180 | epsilon = 0.100 | reward = 5
Episode 15190 | epsilon = 0.100 | reward = -20
Episode 15200 | epsilon = 0.100 | reward = -38
Episode 15210 | epsilon = 0.100 | reward = -14
Episode 15220 | epsilon = 0.100 | reward = -7
Episode 15230 | epsilon = 0.100 | reward = -16
Episode 15240 | epsilon = 0.100 | reward = -4
Episode 15250 | epsilon = 0.100 | reward = -14
Episode 15260 | epsilon = 0.100 | reward = -5
Episode 15270 | epsilon =

Episode 17020 | epsilon = 0.100 | reward = -29
Episode 17030 | epsilon = 0.100 | reward = -35
Episode 17040 | epsilon = 0.100 | reward = -1
Episode 17050 | epsilon = 0.100 | reward = -28
Episode 17060 | epsilon = 0.100 | reward = -5
Episode 17070 | epsilon = 0.100 | reward = -1
Episode 17080 | epsilon = 0.100 | reward = -8
Episode 17090 | epsilon = 0.100 | reward = -13
Episode 17100 | epsilon = 0.100 | reward = -14
Episode 17110 | epsilon = 0.100 | reward = -38
Episode 17120 | epsilon = 0.100 | reward = 5
Episode 17130 | epsilon = 0.100 | reward = -4
Episode 17140 | epsilon = 0.100 | reward = -29
Episode 17150 | epsilon = 0.100 | reward = -8
Episode 17160 | epsilon = 0.100 | reward = -56
Episode 17170 | epsilon = 0.100 | reward = -8
Episode 17180 | epsilon = 0.100 | reward = -17
Episode 17190 | epsilon = 0.100 | reward = -4
Episode 17200 | epsilon = 0.100 | reward = 2
Episode 17210 | epsilon = 0.100 | reward = -14
Episode 17220 | epsilon = 0.100 | reward = -20
Episode 17230 | epsilon =

Episode 18790 | epsilon = 0.100 | reward = 5
Episode 18800 | epsilon = 0.100 | reward = -8
Episode 18810 | epsilon = 0.100 | reward = -8
Episode 18820 | epsilon = 0.100 | reward = -56
Episode 18830 | epsilon = 0.100 | reward = 2
Episode 18840 | epsilon = 0.100 | reward = 2
Episode 18850 | epsilon = 0.100 | reward = -5
Episode 18860 | epsilon = 0.100 | reward = -8
Episode 18870 | epsilon = 0.100 | reward = -7
Episode 18880 | epsilon = 0.100 | reward = -8
Episode 18890 | epsilon = 0.100 | reward = -41
Episode 18900 | epsilon = 0.100 | reward = -4
Episode 18910 | epsilon = 0.100 | reward = 2
Episode 18920 | epsilon = 0.100 | reward = -29
Episode 18930 | epsilon = 0.100 | reward = -8
Episode 18940 | epsilon = 0.100 | reward = -20
Episode 18950 | epsilon = 0.100 | reward = 5
Episode 18960 | epsilon = 0.100 | reward = -23
Episode 18970 | epsilon = 0.100 | reward = -28
Episode 18980 | epsilon = 0.100 | reward = 5
Episode 18990 | epsilon = 0.100 | reward = 5
Episode 19000 | epsilon = 0.100 | r

In [10]:
display_tabular(qValues_qLearning_safeMap, safe_map)

Chosen action: 2
Chosen action: 2
Chosen action: 3
Chosen action: 3
Chosen action: 3
Chosen action: 2
Chosen action: 3
Chosen action: 3
Chosen action: 2
Chosen action: 0
Chosen action: 2
Chosen action: 2
Chosen action: 3
Chosen action: 3
Chosen action: 0
Chosen action: 0
Chosen action: 0
Chosen action: 2
Chosen action: 3
Chosen action: 2
Chosen action: 0
Chosen action: 2
Chosen action: 2
Chosen action: 0
Chosen action: 2
Chosen action: 0
Chosen action: 0
Chosen action: 0
Chosen action: 0
Chosen action: 2
Chosen action: 3
Chosen action: 2
Chosen action: 2
Chosen action: 3
Chosen action: 3
Chosen action: 3
Chosen action: 0
Chosen action: 3
Chosen action: 0
Chosen action: 1


### b- Sarsa

In [11]:
gamma = 0.9
learning_rate = 1
max_episodes = 20000

qValues_sarsa_safeMap = sarsa(safe_env, gamma, learning_rate, max_episodes)
for row in qValues_sarsa_safeMap:
    print(["{:.2f}".format(x) for x in row])

Episode 10 | epsilon = 0.914 | reward = -19
Episode 20 | epsilon = 0.826 | reward = -5
Episode 30 | epsilon = 0.747 | reward = -26
Episode 40 | epsilon = 0.676 | reward = -5
Episode 50 | epsilon = 0.611 | reward = -4
Episode 60 | epsilon = 0.553 | reward = -20
Episode 70 | epsilon = 0.500 | reward = -1
Episode 80 | epsilon = 0.452 | reward = -11
Episode 90 | epsilon = 0.409 | reward = -17
Episode 100 | epsilon = 0.370 | reward = -32
Episode 110 | epsilon = 0.334 | reward = -22
Episode 120 | epsilon = 0.302 | reward = 2
Episode 130 | epsilon = 0.273 | reward = 2
Episode 140 | epsilon = 0.247 | reward = -7
Episode 150 | epsilon = 0.224 | reward = 2
Episode 160 | epsilon = 0.202 | reward = 2
Episode 170 | epsilon = 0.183 | reward = -8
Episode 180 | epsilon = 0.165 | reward = -7
Episode 190 | epsilon = 0.150 | reward = -5
Episode 200 | epsilon = 0.135 | reward = -4
Episode 210 | epsilon = 0.122 | reward = 5
Episode 220 | epsilon = 0.111 | reward = -7
Episode 230 | epsilon = 0.100 | reward 

Episode 2160 | epsilon = 0.100 | reward = 5
Episode 2170 | epsilon = 0.100 | reward = -32
Episode 2180 | epsilon = 0.100 | reward = 2
Episode 2190 | epsilon = 0.100 | reward = 2
Episode 2200 | epsilon = 0.100 | reward = -8
Episode 2210 | epsilon = 0.100 | reward = -5
Episode 2220 | epsilon = 0.100 | reward = 5
Episode 2230 | epsilon = 0.100 | reward = -8
Episode 2240 | epsilon = 0.100 | reward = 5
Episode 2250 | epsilon = 0.100 | reward = 5
Episode 2260 | epsilon = 0.100 | reward = -4
Episode 2270 | epsilon = 0.100 | reward = -5
Episode 2280 | epsilon = 0.100 | reward = 2
Episode 2290 | epsilon = 0.100 | reward = 2
Episode 2300 | epsilon = 0.100 | reward = -11
Episode 2310 | epsilon = 0.100 | reward = -14
Episode 2320 | epsilon = 0.100 | reward = -19
Episode 2330 | epsilon = 0.100 | reward = -8
Episode 2340 | epsilon = 0.100 | reward = -10
Episode 2350 | epsilon = 0.100 | reward = 5
Episode 2360 | epsilon = 0.100 | reward = -5
Episode 2370 | epsilon = 0.100 | reward = -23
Episode 2380 

Episode 3990 | epsilon = 0.100 | reward = -5
Episode 4000 | epsilon = 0.100 | reward = 5
Episode 4010 | epsilon = 0.100 | reward = -23
Episode 4020 | epsilon = 0.100 | reward = 2
Episode 4030 | epsilon = 0.100 | reward = 5
Episode 4040 | epsilon = 0.100 | reward = -23
Episode 4050 | epsilon = 0.100 | reward = -5
Episode 4060 | epsilon = 0.100 | reward = 2
Episode 4070 | epsilon = 0.100 | reward = 5
Episode 4080 | epsilon = 0.100 | reward = 5
Episode 4090 | epsilon = 0.100 | reward = -5
Episode 4100 | epsilon = 0.100 | reward = 5
Episode 4110 | epsilon = 0.100 | reward = -1
Episode 4120 | epsilon = 0.100 | reward = -7
Episode 4130 | epsilon = 0.100 | reward = -5
Episode 4140 | epsilon = 0.100 | reward = 5
Episode 4150 | epsilon = 0.100 | reward = 2
Episode 4160 | epsilon = 0.100 | reward = -8
Episode 4170 | epsilon = 0.100 | reward = -1
Episode 4180 | epsilon = 0.100 | reward = -8
Episode 4190 | epsilon = 0.100 | reward = -1
Episode 4200 | epsilon = 0.100 | reward = 5
Episode 4210 | eps

Episode 6050 | epsilon = 0.100 | reward = 2
Episode 6060 | epsilon = 0.100 | reward = -17
Episode 6070 | epsilon = 0.100 | reward = -1
Episode 6080 | epsilon = 0.100 | reward = 5
Episode 6090 | epsilon = 0.100 | reward = 5
Episode 6100 | epsilon = 0.100 | reward = -7
Episode 6110 | epsilon = 0.100 | reward = -8
Episode 6120 | epsilon = 0.100 | reward = -10
Episode 6130 | epsilon = 0.100 | reward = -5
Episode 6140 | epsilon = 0.100 | reward = -8
Episode 6150 | epsilon = 0.100 | reward = -4
Episode 6160 | epsilon = 0.100 | reward = -4
Episode 6170 | epsilon = 0.100 | reward = -1
Episode 6180 | epsilon = 0.100 | reward = -1
Episode 6190 | epsilon = 0.100 | reward = 5
Episode 6200 | epsilon = 0.100 | reward = 2
Episode 6210 | epsilon = 0.100 | reward = -5
Episode 6220 | epsilon = 0.100 | reward = -8
Episode 6230 | epsilon = 0.100 | reward = 2
Episode 6240 | epsilon = 0.100 | reward = -4
Episode 6250 | epsilon = 0.100 | reward = 2
Episode 6260 | epsilon = 0.100 | reward = 2
Episode 6270 | e

Episode 8100 | epsilon = 0.100 | reward = -5
Episode 8110 | epsilon = 0.100 | reward = -5
Episode 8120 | epsilon = 0.100 | reward = -5
Episode 8130 | epsilon = 0.100 | reward = 2
Episode 8140 | epsilon = 0.100 | reward = -10
Episode 8150 | epsilon = 0.100 | reward = -20
Episode 8160 | epsilon = 0.100 | reward = -4
Episode 8170 | epsilon = 0.100 | reward = -13
Episode 8180 | epsilon = 0.100 | reward = 5
Episode 8190 | epsilon = 0.100 | reward = 5
Episode 8200 | epsilon = 0.100 | reward = -1
Episode 8210 | epsilon = 0.100 | reward = 2
Episode 8220 | epsilon = 0.100 | reward = 2
Episode 8230 | epsilon = 0.100 | reward = -4
Episode 8240 | epsilon = 0.100 | reward = 5
Episode 8250 | epsilon = 0.100 | reward = -11
Episode 8260 | epsilon = 0.100 | reward = 5
Episode 8270 | epsilon = 0.100 | reward = -11
Episode 8280 | epsilon = 0.100 | reward = -7
Episode 8290 | epsilon = 0.100 | reward = 5
Episode 8300 | epsilon = 0.100 | reward = -4
Episode 8310 | epsilon = 0.100 | reward = -17
Episode 8320

Episode 10040 | epsilon = 0.100 | reward = -5
Episode 10050 | epsilon = 0.100 | reward = 2
Episode 10060 | epsilon = 0.100 | reward = -8
Episode 10070 | epsilon = 0.100 | reward = -4
Episode 10080 | epsilon = 0.100 | reward = -1
Episode 10090 | epsilon = 0.100 | reward = -31
Episode 10100 | epsilon = 0.100 | reward = -32
Episode 10110 | epsilon = 0.100 | reward = 5
Episode 10120 | epsilon = 0.100 | reward = -8
Episode 10130 | epsilon = 0.100 | reward = 5
Episode 10140 | epsilon = 0.100 | reward = -1
Episode 10150 | epsilon = 0.100 | reward = -5
Episode 10160 | epsilon = 0.100 | reward = 2
Episode 10170 | epsilon = 0.100 | reward = 2
Episode 10180 | epsilon = 0.100 | reward = -4
Episode 10190 | epsilon = 0.100 | reward = -8
Episode 10200 | epsilon = 0.100 | reward = 5
Episode 10210 | epsilon = 0.100 | reward = 2
Episode 10220 | epsilon = 0.100 | reward = -11
Episode 10230 | epsilon = 0.100 | reward = -4
Episode 10240 | epsilon = 0.100 | reward = -7
Episode 10250 | epsilon = 0.100 | rewa

Episode 12220 | epsilon = 0.100 | reward = -4
Episode 12230 | epsilon = 0.100 | reward = -10
Episode 12240 | epsilon = 0.100 | reward = -5
Episode 12250 | epsilon = 0.100 | reward = -7
Episode 12260 | epsilon = 0.100 | reward = 2
Episode 12270 | epsilon = 0.100 | reward = 2
Episode 12280 | epsilon = 0.100 | reward = 5
Episode 12290 | epsilon = 0.100 | reward = -4
Episode 12300 | epsilon = 0.100 | reward = -1
Episode 12310 | epsilon = 0.100 | reward = -8
Episode 12320 | epsilon = 0.100 | reward = -1
Episode 12330 | epsilon = 0.100 | reward = 5
Episode 12340 | epsilon = 0.100 | reward = -1
Episode 12350 | epsilon = 0.100 | reward = -17
Episode 12360 | epsilon = 0.100 | reward = -4
Episode 12370 | epsilon = 0.100 | reward = -1
Episode 12380 | epsilon = 0.100 | reward = -4
Episode 12390 | epsilon = 0.100 | reward = -7
Episode 12400 | epsilon = 0.100 | reward = -8
Episode 12410 | epsilon = 0.100 | reward = -4
Episode 12420 | epsilon = 0.100 | reward = -10
Episode 12430 | epsilon = 0.100 | r

Episode 14290 | epsilon = 0.100 | reward = -11
Episode 14300 | epsilon = 0.100 | reward = -5
Episode 14310 | epsilon = 0.100 | reward = -5
Episode 14320 | epsilon = 0.100 | reward = 2
Episode 14330 | epsilon = 0.100 | reward = -1
Episode 14340 | epsilon = 0.100 | reward = 5
Episode 14350 | epsilon = 0.100 | reward = -1
Episode 14360 | epsilon = 0.100 | reward = -1
Episode 14370 | epsilon = 0.100 | reward = 5
Episode 14380 | epsilon = 0.100 | reward = -4
Episode 14390 | epsilon = 0.100 | reward = -4
Episode 14400 | epsilon = 0.100 | reward = 5
Episode 14410 | epsilon = 0.100 | reward = 5
Episode 14420 | epsilon = 0.100 | reward = -5
Episode 14430 | epsilon = 0.100 | reward = 5
Episode 14440 | epsilon = 0.100 | reward = -5
Episode 14450 | epsilon = 0.100 | reward = -14
Episode 14460 | epsilon = 0.100 | reward = -5
Episode 14470 | epsilon = 0.100 | reward = -5
Episode 14480 | epsilon = 0.100 | reward = 5
Episode 14490 | epsilon = 0.100 | reward = -5
Episode 14500 | epsilon = 0.100 | rewar

Episode 16380 | epsilon = 0.100 | reward = 2
Episode 16390 | epsilon = 0.100 | reward = 2
Episode 16400 | epsilon = 0.100 | reward = 5
Episode 16410 | epsilon = 0.100 | reward = 5
Episode 16420 | epsilon = 0.100 | reward = -10
Episode 16430 | epsilon = 0.100 | reward = 5
Episode 16440 | epsilon = 0.100 | reward = 2
Episode 16450 | epsilon = 0.100 | reward = -8
Episode 16460 | epsilon = 0.100 | reward = 2
Episode 16470 | epsilon = 0.100 | reward = -5
Episode 16480 | epsilon = 0.100 | reward = 5
Episode 16490 | epsilon = 0.100 | reward = -5
Episode 16500 | epsilon = 0.100 | reward = -11
Episode 16510 | epsilon = 0.100 | reward = -5
Episode 16520 | epsilon = 0.100 | reward = 5
Episode 16530 | epsilon = 0.100 | reward = -5
Episode 16540 | epsilon = 0.100 | reward = -5
Episode 16550 | epsilon = 0.100 | reward = 2
Episode 16560 | epsilon = 0.100 | reward = -10
Episode 16570 | epsilon = 0.100 | reward = -23
Episode 16580 | epsilon = 0.100 | reward = -5
Episode 16590 | epsilon = 0.100 | reward

Episode 18190 | epsilon = 0.100 | reward = -4
Episode 18200 | epsilon = 0.100 | reward = 2
Episode 18210 | epsilon = 0.100 | reward = -11
Episode 18220 | epsilon = 0.100 | reward = -16
Episode 18230 | epsilon = 0.100 | reward = 5
Episode 18240 | epsilon = 0.100 | reward = -1
Episode 18250 | epsilon = 0.100 | reward = -5
Episode 18260 | epsilon = 0.100 | reward = -7
Episode 18270 | epsilon = 0.100 | reward = 2
Episode 18280 | epsilon = 0.100 | reward = -1
Episode 18290 | epsilon = 0.100 | reward = 5
Episode 18300 | epsilon = 0.100 | reward = 5
Episode 18310 | epsilon = 0.100 | reward = -8
Episode 18320 | epsilon = 0.100 | reward = -1
Episode 18330 | epsilon = 0.100 | reward = -4
Episode 18340 | epsilon = 0.100 | reward = 5
Episode 18350 | epsilon = 0.100 | reward = 5
Episode 18360 | epsilon = 0.100 | reward = -1
Episode 18370 | epsilon = 0.100 | reward = 5
Episode 18380 | epsilon = 0.100 | reward = 5
Episode 18390 | epsilon = 0.100 | reward = -1
Episode 18400 | epsilon = 0.100 | reward 

In [12]:
display_tabular(qValues_sarsa_safeMap, safe_map)

Chosen action: 1
Chosen action: 1
Chosen action: 2
Chosen action: 1
Chosen action: 0
Chosen action: 0
Chosen action: 0
Chosen action: 1


# Premier Environnement difficile

In [13]:
hard_map1 = [
    "SFFFFG",
    "FFFHHF",
    "FFFFHF",
    "FFHGHF",
    "FFFFFF",
    "FFFFFG"
]

hard_env1 = gym.make("FrozenLake-v1", desc=hard_map1, is_slippery=True)
hard_env1 = CustomFrozenLake(hard_env1)

### a- Q-learning

In [14]:
# Il faut faire une recherche en grille ou aléatoire pour trouver les bons hyperparams ?
gamma = 0.99 
learning_rate = 1
max_episodes = 20000

qValues_qLearning_hardMap1 = q_learning(hard_env1, gamma, learning_rate, max_episodes)
for row in qValues_qLearning_hardMap1: # Pour afficher les valeurs des actions dans une format lisible
    print(["{:.2f}".format(x) for x in row])

Episode 10 | epsilon = 0.914 | reward = -71
Episode 20 | epsilon = 0.826 | reward = -23
Episode 30 | epsilon = 0.747 | reward = -5
Episode 40 | epsilon = 0.676 | reward = -14
Episode 50 | epsilon = 0.611 | reward = -8
Episode 60 | epsilon = 0.553 | reward = -17
Episode 70 | epsilon = 0.500 | reward = -11
Episode 80 | epsilon = 0.452 | reward = -37
Episode 90 | epsilon = 0.409 | reward = -8
Episode 100 | epsilon = 0.370 | reward = -14
Episode 110 | epsilon = 0.334 | reward = -14
Episode 120 | epsilon = 0.302 | reward = -10
Episode 130 | epsilon = 0.273 | reward = -4
Episode 140 | epsilon = 0.247 | reward = -11
Episode 150 | epsilon = 0.224 | reward = -5
Episode 160 | epsilon = 0.202 | reward = 2
Episode 170 | epsilon = 0.183 | reward = -34
Episode 180 | epsilon = 0.165 | reward = -5
Episode 190 | epsilon = 0.150 | reward = -26
Episode 200 | epsilon = 0.135 | reward = -7
Episode 210 | epsilon = 0.122 | reward = -8
Episode 220 | epsilon = 0.111 | reward = -5
Episode 230 | epsilon = 0.100 

Episode 2020 | epsilon = 0.100 | reward = -14
Episode 2030 | epsilon = 0.100 | reward = -4
Episode 2040 | epsilon = 0.100 | reward = -4
Episode 2050 | epsilon = 0.100 | reward = -1
Episode 2060 | epsilon = 0.100 | reward = 2
Episode 2070 | epsilon = 0.100 | reward = -24
Episode 2080 | epsilon = 0.100 | reward = -23
Episode 2090 | epsilon = 0.100 | reward = -44
Episode 2100 | epsilon = 0.100 | reward = -32
Episode 2110 | epsilon = 0.100 | reward = -14
Episode 2120 | epsilon = 0.100 | reward = -5
Episode 2130 | epsilon = 0.100 | reward = -20
Episode 2140 | epsilon = 0.100 | reward = -14
Episode 2150 | epsilon = 0.100 | reward = -1
Episode 2160 | epsilon = 0.100 | reward = -5
Episode 2170 | epsilon = 0.100 | reward = -20
Episode 2180 | epsilon = 0.100 | reward = -4
Episode 2190 | epsilon = 0.100 | reward = -8
Episode 2200 | epsilon = 0.100 | reward = -14
Episode 2210 | epsilon = 0.100 | reward = -5
Episode 2220 | epsilon = 0.100 | reward = 5
Episode 2230 | epsilon = 0.100 | reward = -29
E

Episode 3910 | epsilon = 0.100 | reward = -11
Episode 3920 | epsilon = 0.100 | reward = -29
Episode 3930 | epsilon = 0.100 | reward = 2
Episode 3940 | epsilon = 0.100 | reward = -48
Episode 3950 | epsilon = 0.100 | reward = -20
Episode 3960 | epsilon = 0.100 | reward = -8
Episode 3970 | epsilon = 0.100 | reward = -16
Episode 3980 | epsilon = 0.100 | reward = -1
Episode 3990 | epsilon = 0.100 | reward = -11
Episode 4000 | epsilon = 0.100 | reward = -11
Episode 4010 | epsilon = 0.100 | reward = -1
Episode 4020 | epsilon = 0.100 | reward = -8
Episode 4030 | epsilon = 0.100 | reward = -4
Episode 4040 | epsilon = 0.100 | reward = -20
Episode 4050 | epsilon = 0.100 | reward = -5
Episode 4060 | epsilon = 0.100 | reward = -1
Episode 4070 | epsilon = 0.100 | reward = -5
Episode 4080 | epsilon = 0.100 | reward = -23
Episode 4090 | epsilon = 0.100 | reward = -23
Episode 4100 | epsilon = 0.100 | reward = -8
Episode 4110 | epsilon = 0.100 | reward = -5
Episode 4120 | epsilon = 0.100 | reward = -29


Episode 5740 | epsilon = 0.100 | reward = -26
Episode 5750 | epsilon = 0.100 | reward = -21
Episode 5760 | epsilon = 0.100 | reward = -23
Episode 5770 | epsilon = 0.100 | reward = -10
Episode 5780 | epsilon = 0.100 | reward = -1
Episode 5790 | epsilon = 0.100 | reward = -17
Episode 5800 | epsilon = 0.100 | reward = -5
Episode 5810 | epsilon = 0.100 | reward = -23
Episode 5820 | epsilon = 0.100 | reward = -41
Episode 5830 | epsilon = 0.100 | reward = 5
Episode 5840 | epsilon = 0.100 | reward = -1
Episode 5850 | epsilon = 0.100 | reward = -4
Episode 5860 | epsilon = 0.100 | reward = 5
Episode 5870 | epsilon = 0.100 | reward = -13
Episode 5880 | epsilon = 0.100 | reward = -29
Episode 5890 | epsilon = 0.100 | reward = -17
Episode 5900 | epsilon = 0.100 | reward = -13
Episode 5910 | epsilon = 0.100 | reward = -28
Episode 5920 | epsilon = 0.100 | reward = -11
Episode 5930 | epsilon = 0.100 | reward = -11
Episode 5940 | epsilon = 0.100 | reward = -20
Episode 5950 | epsilon = 0.100 | reward = 

Episode 7660 | epsilon = 0.100 | reward = -13
Episode 7670 | epsilon = 0.100 | reward = -14
Episode 7680 | epsilon = 0.100 | reward = -10
Episode 7690 | epsilon = 0.100 | reward = -8
Episode 7700 | epsilon = 0.100 | reward = -14
Episode 7710 | epsilon = 0.100 | reward = 5
Episode 7720 | epsilon = 0.100 | reward = -21
Episode 7730 | epsilon = 0.100 | reward = -29
Episode 7740 | epsilon = 0.100 | reward = -5
Episode 7750 | epsilon = 0.100 | reward = -8
Episode 7760 | epsilon = 0.100 | reward = -20
Episode 7770 | epsilon = 0.100 | reward = -13
Episode 7780 | epsilon = 0.100 | reward = -17
Episode 7790 | epsilon = 0.100 | reward = -5
Episode 7800 | epsilon = 0.100 | reward = -8
Episode 7810 | epsilon = 0.100 | reward = -39
Episode 7820 | epsilon = 0.100 | reward = -7
Episode 7830 | epsilon = 0.100 | reward = -8
Episode 7840 | epsilon = 0.100 | reward = -8
Episode 7850 | epsilon = 0.100 | reward = -19
Episode 7860 | epsilon = 0.100 | reward = -5
Episode 7870 | epsilon = 0.100 | reward = -25

...

Episode 11240 | epsilon = 0.100 | reward = -11
Episode 11250 | epsilon = 0.100 | reward = -21
Episode 11260 | epsilon = 0.100 | reward = 2
Episode 11270 | epsilon = 0.100 | reward = -14
Episode 11280 | epsilon = 0.100 | reward = 2
Episode 11290 | epsilon = 0.100 | reward = -1
Episode 11300 | epsilon = 0.100 | reward = 2
Episode 11310 | epsilon = 0.100 | reward = -17
Episode 11320 | epsilon = 0.100 | reward = -16
Episode 11330 | epsilon = 0.100 | reward = -23
Episode 11340 | epsilon = 0.100 | reward = 5
Episode 11350 | epsilon = 0.100 | reward = -40
Episode 11360 | epsilon = 0.100 | reward = -8
Episode 11370 | epsilon = 0.100 | reward = -14
Episode 11380 | epsilon = 0.100 | reward = -10
Episode 11390 | epsilon = 0.100 | reward = -1
Episode 11400 | epsilon = 0.100 | reward = -17
Episode 11410 | epsilon = 0.100 | reward = -20
Episode 11420 | epsilon = 0.100 | reward = -22
Episode 11430 | epsilon = 0.100 | reward = -11
Episode 11440 | epsilon = 0.100 | reward = -4
Episode 11450 | epsilon =

Episode 13010 | epsilon = 0.100 | reward = -20
Episode 13020 | epsilon = 0.100 | reward = -14
Episode 13030 | epsilon = 0.100 | reward = -5
Episode 13040 | epsilon = 0.100 | reward = -11
Episode 13050 | epsilon = 0.100 | reward = -5
Episode 13060 | epsilon = 0.100 | reward = -10
Episode 13070 | epsilon = 0.100 | reward = -5
Episode 13080 | epsilon = 0.100 | reward = -5
Episode 13090 | epsilon = 0.100 | reward = -26
Episode 13100 | epsilon = 0.100 | reward = -4
Episode 13110 | epsilon = 0.100 | reward = -20
Episode 13120 | epsilon = 0.100 | reward = -11
Episode 13130 | epsilon = 0.100 | reward = -35
Episode 13140 | epsilon = 0.100 | reward = -5
Episode 13150 | epsilon = 0.100 | reward = -17
Episode 13160 | epsilon = 0.100 | reward = 5
Episode 13170 | epsilon = 0.100 | reward = -20
Episode 13180 | epsilon = 0.100 | reward = -20
Episode 13190 | epsilon = 0.100 | reward = -8
Episode 13200 | epsilon = 0.100 | reward = -7
Episode 13210 | epsilon = 0.100 | reward = -23
Episode 13220 | epsilon

Episode 14870 | epsilon = 0.100 | reward = -1
Episode 14880 | epsilon = 0.100 | reward = -11
Episode 14890 | epsilon = 0.100 | reward = 2
Episode 14900 | epsilon = 0.100 | reward = -13
Episode 14910 | epsilon = 0.100 | reward = -14
Episode 14920 | epsilon = 0.100 | reward = -23
Episode 14930 | epsilon = 0.100 | reward = -1
Episode 14940 | epsilon = 0.100 | reward = -11
Episode 14950 | epsilon = 0.100 | reward = -44
Episode 14960 | epsilon = 0.100 | reward = -7
Episode 14970 | epsilon = 0.100 | reward = -8
Episode 14980 | epsilon = 0.100 | reward = -1
Episode 14990 | epsilon = 0.100 | reward = -25
Episode 15000 | epsilon = 0.100 | reward = -39
Episode 15010 | epsilon = 0.100 | reward = -8
Episode 15020 | epsilon = 0.100 | reward = -35
Episode 15030 | epsilon = 0.100 | reward = -17
Episode 15040 | epsilon = 0.100 | reward = -1
Episode 15050 | epsilon = 0.100 | reward = -8
Episode 15060 | epsilon = 0.100 | reward = -14
Episode 15070 | epsilon = 0.100 | reward = -7
Episode 15080 | epsilon 

Episode 16700 | epsilon = 0.100 | reward = -35
Episode 16710 | epsilon = 0.100 | reward = -7
Episode 16720 | epsilon = 0.100 | reward = -41
Episode 16730 | epsilon = 0.100 | reward = -14
Episode 16740 | epsilon = 0.100 | reward = -5
Episode 16750 | epsilon = 0.100 | reward = -5
Episode 16760 | epsilon = 0.100 | reward = -13
Episode 16770 | epsilon = 0.100 | reward = -11
Episode 16780 | epsilon = 0.100 | reward = -11
Episode 16790 | epsilon = 0.100 | reward = -5
Episode 16800 | epsilon = 0.100 | reward = -1
Episode 16810 | epsilon = 0.100 | reward = -1
Episode 16820 | epsilon = 0.100 | reward = -10
Episode 16830 | epsilon = 0.100 | reward = -43
Episode 16840 | epsilon = 0.100 | reward = -20
Episode 16850 | epsilon = 0.100 | reward = -10
Episode 16860 | epsilon = 0.100 | reward = -8
Episode 16870 | epsilon = 0.100 | reward = -5
Episode 16880 | epsilon = 0.100 | reward = -23
Episode 16890 | epsilon = 0.100 | reward = -56
Episode 16900 | epsilon = 0.100 | reward = -8
Episode 16910 | epsilo

Episode 18540 | epsilon = 0.100 | reward = -22
Episode 18550 | epsilon = 0.100 | reward = -10
Episode 18560 | epsilon = 0.100 | reward = -10
Episode 18570 | epsilon = 0.100 | reward = -5
Episode 18580 | epsilon = 0.100 | reward = -14
Episode 18590 | epsilon = 0.100 | reward = -5
Episode 18600 | epsilon = 0.100 | reward = -8
Episode 18610 | epsilon = 0.100 | reward = -13
Episode 18620 | epsilon = 0.100 | reward = -27
Episode 18630 | epsilon = 0.100 | reward = -8
Episode 18640 | epsilon = 0.100 | reward = 5
Episode 18650 | epsilon = 0.100 | reward = -63
Episode 18660 | epsilon = 0.100 | reward = -5
Episode 18670 | epsilon = 0.100 | reward = -14
Episode 18680 | epsilon = 0.100 | reward = -5
Episode 18690 | epsilon = 0.100 | reward = -36
Episode 18700 | epsilon = 0.100 | reward = -11
Episode 18710 | epsilon = 0.100 | reward = -20
Episode 18720 | epsilon = 0.100 | reward = -8
Episode 18730 | epsilon = 0.100 | reward = -14
Episode 18740 | epsilon = 0.100 | reward = 2
Episode 18750 | epsilon 

In [15]:
display_tabular(qValues_qLearning_hardMap1, hard_map1)

Chosen action: 1
Chosen action: 2
Chosen action: 0
Chosen action: 0
Chosen action: 0
Chosen action: 0
Chosen action: 2
Chosen action: 2
Chosen action: 2
Chosen action: 3
Chosen action: 3
Chosen action: 3
Chosen action: 3
Chosen action: 3
Chosen action: 3
Chosen action: 1
Chosen action: 3
Chosen action: 3
Chosen action: 1
Chosen action: 3
Chosen action: 3
Chosen action: 3
Chosen action: 3
Chosen action: 3


### b- Sarsa

In [16]:
gamma = 0.9
learning_rate = 1
max_episodes = 20000

qValues_sarsa_hardMap1 = sarsa(hard_env1, gamma, learning_rate, max_episodes)
for row in qValues_sarsa_hardMap1:
    print(["{:.2f}".format(x) for x in row])

Episode 10 | epsilon = 0.914 | reward = -20
Episode 20 | epsilon = 0.826 | reward = -11
Episode 30 | epsilon = 0.747 | reward = -5
Episode 40 | epsilon = 0.676 | reward = -14
Episode 50 | epsilon = 0.611 | reward = -17
Episode 60 | epsilon = 0.553 | reward = -22
Episode 70 | epsilon = 0.500 | reward = -40
Episode 80 | epsilon = 0.452 | reward = -8
Episode 90 | epsilon = 0.409 | reward = -5
Episode 100 | epsilon = 0.370 | reward = -8
Episode 110 | epsilon = 0.334 | reward = -11
Episode 120 | epsilon = 0.302 | reward = -8
Episode 130 | epsilon = 0.273 | reward = -32
Episode 140 | epsilon = 0.247 | reward = -20
Episode 150 | epsilon = 0.224 | reward = 5
Episode 160 | epsilon = 0.202 | reward = -11
Episode 170 | epsilon = 0.183 | reward = -8
Episode 180 | epsilon = 0.165 | reward = -11
Episode 190 | epsilon = 0.150 | reward = 2
Episode 200 | epsilon = 0.135 | reward = -1
Episode 210 | epsilon = 0.122 | reward = -10
Episode 220 | epsilon = 0.111 | reward = -7
Episode 230 | epsilon = 0.100 |

Episode 1970 | epsilon = 0.100 | reward = -7
Episode 1980 | epsilon = 0.100 | reward = -1
Episode 1990 | epsilon = 0.100 | reward = -8
Episode 2000 | epsilon = 0.100 | reward = 2
Episode 2010 | epsilon = 0.100 | reward = -14
Episode 2020 | epsilon = 0.100 | reward = 2
Episode 2030 | epsilon = 0.100 | reward = -17
Episode 2040 | epsilon = 0.100 | reward = -7
Episode 2050 | epsilon = 0.100 | reward = 5
Episode 2060 | epsilon = 0.100 | reward = -8
Episode 2070 | epsilon = 0.100 | reward = -22
Episode 2080 | epsilon = 0.100 | reward = -8
Episode 2090 | epsilon = 0.100 | reward = -5
Episode 2100 | epsilon = 0.100 | reward = -1
Episode 2110 | epsilon = 0.100 | reward = -11
Episode 2120 | epsilon = 0.100 | reward = -19
Episode 2130 | epsilon = 0.100 | reward = -5
Episode 2140 | epsilon = 0.100 | reward = -1
Episode 2150 | epsilon = 0.100 | reward = -4
Episode 2160 | epsilon = 0.100 | reward = -5
Episode 2170 | epsilon = 0.100 | reward = -11
Episode 2180 | epsilon = 0.100 | reward = -4
Episode

Episode 3850 | epsilon = 0.100 | reward = -17
Episode 3860 | epsilon = 0.100 | reward = -26
Episode 3870 | epsilon = 0.100 | reward = -7
Episode 3880 | epsilon = 0.100 | reward = 2
Episode 3890 | epsilon = 0.100 | reward = -26
Episode 3900 | epsilon = 0.100 | reward = -5
Episode 3910 | epsilon = 0.100 | reward = -20
Episode 3920 | epsilon = 0.100 | reward = -4
Episode 3930 | epsilon = 0.100 | reward = -1
Episode 3940 | epsilon = 0.100 | reward = 5
Episode 3950 | epsilon = 0.100 | reward = 2
Episode 3960 | epsilon = 0.100 | reward = 5
Episode 3970 | epsilon = 0.100 | reward = -5
Episode 3980 | epsilon = 0.100 | reward = -17
Episode 3990 | epsilon = 0.100 | reward = -29
Episode 4000 | epsilon = 0.100 | reward = 2
Episode 4010 | epsilon = 0.100 | reward = -17
Episode 4020 | epsilon = 0.100 | reward = -11
Episode 4030 | epsilon = 0.100 | reward = -13
Episode 4040 | epsilon = 0.100 | reward = -4
Episode 4050 | epsilon = 0.100 | reward = -8
Episode 4060 | epsilon = 0.100 | reward = 2
Episode

Episode 5830 | epsilon = 0.100 | reward = -5
Episode 5840 | epsilon = 0.100 | reward = 5
Episode 5850 | epsilon = 0.100 | reward = -28
Episode 5860 | epsilon = 0.100 | reward = -25
Episode 5870 | epsilon = 0.100 | reward = -1
Episode 5880 | epsilon = 0.100 | reward = 2
Episode 5890 | epsilon = 0.100 | reward = -5
Episode 5900 | epsilon = 0.100 | reward = -4
Episode 5910 | epsilon = 0.100 | reward = -5
Episode 5920 | epsilon = 0.100 | reward = 5
Episode 5930 | epsilon = 0.100 | reward = -7
Episode 5940 | epsilon = 0.100 | reward = 5
Episode 5950 | epsilon = 0.100 | reward = -1
Episode 5960 | epsilon = 0.100 | reward = -17
Episode 5970 | epsilon = 0.100 | reward = -11
Episode 5980 | epsilon = 0.100 | reward = -8
Episode 5990 | epsilon = 0.100 | reward = -1
Episode 6000 | epsilon = 0.100 | reward = -11
Episode 6010 | epsilon = 0.100 | reward = 5
Episode 6020 | epsilon = 0.100 | reward = -11
Episode 6030 | epsilon = 0.100 | reward = -5
Episode 6040 | epsilon = 0.100 | reward = -1
Episode 6

Episode 7900 | epsilon = 0.100 | reward = -35
Episode 7910 | epsilon = 0.100 | reward = 5
Episode 7920 | epsilon = 0.100 | reward = -11
Episode 7930 | epsilon = 0.100 | reward = -14
Episode 7940 | epsilon = 0.100 | reward = -11
Episode 7950 | epsilon = 0.100 | reward = -5
Episode 7960 | epsilon = 0.100 | reward = -8
Episode 7970 | epsilon = 0.100 | reward = -5
Episode 7980 | epsilon = 0.100 | reward = -5
Episode 7990 | epsilon = 0.100 | reward = -5
Episode 8000 | epsilon = 0.100 | reward = -8
Episode 8010 | epsilon = 0.100 | reward = -14
Episode 8020 | epsilon = 0.100 | reward = 5
Episode 8030 | epsilon = 0.100 | reward = 5
Episode 8040 | epsilon = 0.100 | reward = -14
Episode 8050 | epsilon = 0.100 | reward = 2
Episode 8060 | epsilon = 0.100 | reward = -4
Episode 8070 | epsilon = 0.100 | reward = -7
Episode 8080 | epsilon = 0.100 | reward = 2
Episode 8090 | epsilon = 0.100 | reward = -8
Episode 8100 | epsilon = 0.100 | reward = -38
Episode 8110 | epsilon = 0.100 | reward = -5
Episode 

...

Episode 11800 | epsilon = 0.100 | reward = -8
Episode 11810 | epsilon = 0.100 | reward = -5
Episode 11820 | epsilon = 0.100 | reward = -20
Episode 11830 | epsilon = 0.100 | reward = -5
Episode 11840 | epsilon = 0.100 | reward = -3
Episode 11850 | epsilon = 0.100 | reward = -1
Episode 11860 | epsilon = 0.100 | reward = -5
Episode 11870 | epsilon = 0.100 | reward = -4
Episode 11880 | epsilon = 0.100 | reward = -14
Episode 11890 | epsilon = 0.100 | reward = -1
Episode 11900 | epsilon = 0.100 | reward = -21
Episode 11910 | epsilon = 0.100 | reward = -24
Episode 11920 | epsilon = 0.100 | reward = -20
Episode 11930 | epsilon = 0.100 | reward = -25
Episode 11940 | epsilon = 0.100 | reward = -5
Episode 11950 | epsilon = 0.100 | reward = -11
Episode 11960 | epsilon = 0.100 | reward = -14
Episode 11970 | epsilon = 0.100 | reward = -1
Episode 11980 | epsilon = 0.100 | reward = -5
Episode 11990 | epsilon = 0.100 | reward = -1
Episode 12000 | epsilon = 0.100 | reward = 5
Episode 12010 | epsilon = 0

Episode 13630 | epsilon = 0.100 | reward = 2
Episode 13640 | epsilon = 0.100 | reward = 5
Episode 13650 | epsilon = 0.100 | reward = 5
Episode 13660 | epsilon = 0.100 | reward = -13
Episode 13670 | epsilon = 0.100 | reward = -4
Episode 13680 | epsilon = 0.100 | reward = -32
Episode 13690 | epsilon = 0.100 | reward = -20
Episode 13700 | epsilon = 0.100 | reward = -8
Episode 13710 | epsilon = 0.100 | reward = -5
Episode 13720 | epsilon = 0.100 | reward = -12
Episode 13730 | epsilon = 0.100 | reward = 2
Episode 13740 | epsilon = 0.100 | reward = -8
Episode 13750 | epsilon = 0.100 | reward = 5
Episode 13760 | epsilon = 0.100 | reward = -8
Episode 13770 | epsilon = 0.100 | reward = -1
Episode 13780 | epsilon = 0.100 | reward = -4
Episode 13790 | epsilon = 0.100 | reward = -5
Episode 13800 | epsilon = 0.100 | reward = -11
Episode 13810 | epsilon = 0.100 | reward = 5
Episode 13820 | epsilon = 0.100 | reward = 5
Episode 13830 | epsilon = 0.100 | reward = 5
Episode 13840 | epsilon = 0.100 | rew

Episode 15510 | epsilon = 0.100 | reward = -4
Episode 15520 | epsilon = 0.100 | reward = 2
Episode 15530 | epsilon = 0.100 | reward = 5
Episode 15540 | epsilon = 0.100 | reward = -5
Episode 15550 | epsilon = 0.100 | reward = -11
Episode 15560 | epsilon = 0.100 | reward = -8
Episode 15570 | epsilon = 0.100 | reward = -1
Episode 15580 | epsilon = 0.100 | reward = 2
Episode 15590 | epsilon = 0.100 | reward = -8
Episode 15600 | epsilon = 0.100 | reward = -5
Episode 15610 | epsilon = 0.100 | reward = -13
Episode 15620 | epsilon = 0.100 | reward = -5
Episode 15630 | epsilon = 0.100 | reward = 2
Episode 15640 | epsilon = 0.100 | reward = 5
Episode 15650 | epsilon = 0.100 | reward = -1
Episode 15660 | epsilon = 0.100 | reward = -7
Episode 15670 | epsilon = 0.100 | reward = -11
Episode 15680 | epsilon = 0.100 | reward = -8
Episode 15690 | epsilon = 0.100 | reward = 2
Episode 15700 | epsilon = 0.100 | reward = -22
Episode 15710 | epsilon = 0.100 | reward = -1
Episode 15720 | epsilon = 0.100 | re

Episode 17450 | epsilon = 0.100 | reward = -10
Episode 17460 | epsilon = 0.100 | reward = -1
Episode 17470 | epsilon = 0.100 | reward = 5
Episode 17480 | epsilon = 0.100 | reward = -20
Episode 17490 | epsilon = 0.100 | reward = -10
Episode 17500 | epsilon = 0.100 | reward = -10
Episode 17510 | epsilon = 0.100 | reward = 2
Episode 17520 | epsilon = 0.100 | reward = -14
Episode 17530 | epsilon = 0.100 | reward = -10
Episode 17540 | epsilon = 0.100 | reward = -1
Episode 17550 | epsilon = 0.100 | reward = -1
Episode 17560 | epsilon = 0.100 | reward = -7
Episode 17570 | epsilon = 0.100 | reward = -11
Episode 17580 | epsilon = 0.100 | reward = -29
Episode 17590 | epsilon = 0.100 | reward = -8
Episode 17600 | epsilon = 0.100 | reward = -11
Episode 17610 | epsilon = 0.100 | reward = -5
Episode 17620 | epsilon = 0.100 | reward = -1
Episode 17630 | epsilon = 0.100 | reward = -4
Episode 17640 | epsilon = 0.100 | reward = 5
Episode 17650 | epsilon = 0.100 | reward = -5
Episode 17660 | epsilon = 0.

Episode 19340 | epsilon = 0.100 | reward = 2
Episode 19350 | epsilon = 0.100 | reward = -8
Episode 19360 | epsilon = 0.100 | reward = 2
Episode 19370 | epsilon = 0.100 | reward = -5
Episode 19380 | epsilon = 0.100 | reward = 5
Episode 19390 | epsilon = 0.100 | reward = -4
Episode 19400 | epsilon = 0.100 | reward = 5
Episode 19410 | epsilon = 0.100 | reward = -7
Episode 19420 | epsilon = 0.100 | reward = -5
Episode 19430 | epsilon = 0.100 | reward = 2
Episode 19440 | epsilon = 0.100 | reward = -8
Episode 19450 | epsilon = 0.100 | reward = -7
Episode 19460 | epsilon = 0.100 | reward = -11
Episode 19470 | epsilon = 0.100 | reward = -10
Episode 19480 | epsilon = 0.100 | reward = -5
Episode 19490 | epsilon = 0.100 | reward = -19
Episode 19500 | epsilon = 0.100 | reward = 5
Episode 19510 | epsilon = 0.100 | reward = -4
Episode 19520 | epsilon = 0.100 | reward = -5
Episode 19530 | epsilon = 0.100 | reward = -16
Episode 19540 | epsilon = 0.100 | reward = -26
Episode 19550 | epsilon = 0.100 | r

In [17]:
display_tabular(qValues_sarsa_hardMap1, hard_map1)

Chosen action: 2
Chosen action: 2
Chosen action: 2
Chosen action: 2
Chosen action: 1
Chosen action: 2
Chosen action: 2
Chosen action: 2
Chosen action: 2
Chosen action: 1
Chosen action: 3
Chosen action: 1
Chosen action: 0
Chosen action: 2
Chosen action: 0
Chosen action: 2
Chosen action: 2
Chosen action: 2
Chosen action: 0
Chosen action: 2
Chosen action: 0
Chosen action: 2
Chosen action: 3
Chosen action: 3
Chosen action: 1
Chosen action: 3
Chosen action: 1
Chosen action: 2


# Deuxième environnement difficile

In [18]:
hard_map2 = [
    "SFFFFF",
    "FFHFHF",
    "FFHGFF",
    "FFFFFH",
    "FFFFFF",
    "HHFHHH"
]

hard_env2 = gym.make("FrozenLake-v1", desc=hard_map2, is_slippery=True)
hard_env2 = CustomFrozenLake(hard_env2)

### a- Q-learning

In [19]:
# Il faut faire une recherche en grille ou aléatoire pour trouver les bons hyperparams ?
gamma = 0.99 
learning_rate = 1
max_episodes = 20000

qValues_qLearning_hardMap2 = q_learning(hard_env2, gamma, learning_rate, max_episodes)
for row in qValues_qLearning_hardMap2: # Pour afficher les valeurs des actions dans une format lisible
    print(["{:.2f}".format(x) for x in row])

Episode 10 | epsilon = 0.914 | reward = -29
Episode 20 | epsilon = 0.826 | reward = -8
Episode 30 | epsilon = 0.747 | reward = -11
Episode 40 | epsilon = 0.676 | reward = -20
Episode 50 | epsilon = 0.611 | reward = -11
Episode 60 | epsilon = 0.553 | reward = -20
Episode 70 | epsilon = 0.500 | reward = -8
Episode 80 | epsilon = 0.452 | reward = -13
Episode 90 | epsilon = 0.409 | reward = -5
Episode 100 | epsilon = 0.370 | reward = -17
Episode 110 | epsilon = 0.334 | reward = -26
Episode 120 | epsilon = 0.302 | reward = -8
Episode 130 | epsilon = 0.273 | reward = 2
Episode 140 | epsilon = 0.247 | reward = -23
Episode 150 | epsilon = 0.224 | reward = -5
Episode 160 | epsilon = 0.202 | reward = -59
Episode 170 | epsilon = 0.183 | reward = -20
Episode 180 | epsilon = 0.165 | reward = -14
Episode 190 | epsilon = 0.150 | reward = -5
Episode 200 | epsilon = 0.135 | reward = -14
Episode 210 | epsilon = 0.122 | reward = -5
Episode 220 | epsilon = 0.111 | reward = -5
Episode 230 | epsilon = 0.100

Episode 1840 | epsilon = 0.100 | reward = -8
Episode 1850 | epsilon = 0.100 | reward = -56
Episode 1860 | epsilon = 0.100 | reward = -38
Episode 1870 | epsilon = 0.100 | reward = -5
Episode 1880 | epsilon = 0.100 | reward = -26
Episode 1890 | epsilon = 0.100 | reward = -8
Episode 1900 | epsilon = 0.100 | reward = -11
Episode 1910 | epsilon = 0.100 | reward = -38
Episode 1920 | epsilon = 0.100 | reward = -11
Episode 1930 | epsilon = 0.100 | reward = -11
Episode 1940 | epsilon = 0.100 | reward = -8
Episode 1950 | epsilon = 0.100 | reward = -5
Episode 1960 | epsilon = 0.100 | reward = -8
Episode 1970 | epsilon = 0.100 | reward = -5
Episode 1980 | epsilon = 0.100 | reward = -5
Episode 1990 | epsilon = 0.100 | reward = -32
Episode 2000 | epsilon = 0.100 | reward = -8
Episode 2010 | epsilon = 0.100 | reward = -14
Episode 2020 | epsilon = 0.100 | reward = -17
Episode 2030 | epsilon = 0.100 | reward = -17
Episode 2040 | epsilon = 0.100 | reward = -26
Episode 2050 | epsilon = 0.100 | reward = -

Episode 3680 | epsilon = 0.100 | reward = -23
Episode 3690 | epsilon = 0.100 | reward = -8
Episode 3700 | epsilon = 0.100 | reward = -35
Episode 3710 | epsilon = 0.100 | reward = -5
Episode 3720 | epsilon = 0.100 | reward = -8
Episode 3730 | epsilon = 0.100 | reward = -14
Episode 3740 | epsilon = 0.100 | reward = 5
Episode 3750 | epsilon = 0.100 | reward = -5
Episode 3760 | epsilon = 0.100 | reward = -11
Episode 3770 | epsilon = 0.100 | reward = -20
Episode 3780 | epsilon = 0.100 | reward = -14
Episode 3790 | epsilon = 0.100 | reward = -20
Episode 3800 | epsilon = 0.100 | reward = -11
Episode 3810 | epsilon = 0.100 | reward = -8
Episode 3820 | epsilon = 0.100 | reward = -5
Episode 3830 | epsilon = 0.100 | reward = -17
Episode 3840 | epsilon = 0.100 | reward = -20
Episode 3850 | epsilon = 0.100 | reward = -11
Episode 3860 | epsilon = 0.100 | reward = -8
Episode 3870 | epsilon = 0.100 | reward = -17
Episode 3880 | epsilon = 0.100 | reward = -5
Episode 3890 | epsilon = 0.100 | reward = -5

Episode 5580 | epsilon = 0.100 | reward = -17
Episode 5590 | epsilon = 0.100 | reward = -5
Episode 5600 | epsilon = 0.100 | reward = -26
Episode 5610 | epsilon = 0.100 | reward = -11
Episode 5620 | epsilon = 0.100 | reward = -5
Episode 5630 | epsilon = 0.100 | reward = -17
Episode 5640 | epsilon = 0.100 | reward = -5
Episode 5650 | epsilon = 0.100 | reward = -14
Episode 5660 | epsilon = 0.100 | reward = -14
Episode 5670 | epsilon = 0.100 | reward = -68
Episode 5680 | epsilon = 0.100 | reward = -8
Episode 5690 | epsilon = 0.100 | reward = -11
Episode 5700 | epsilon = 0.100 | reward = -29
Episode 5710 | epsilon = 0.100 | reward = -17
Episode 5720 | epsilon = 0.100 | reward = -20
Episode 5730 | epsilon = 0.100 | reward = -26
Episode 5740 | epsilon = 0.100 | reward = -47
Episode 5750 | epsilon = 0.100 | reward = -5
Episode 5760 | epsilon = 0.100 | reward = -47
Episode 5770 | epsilon = 0.100 | reward = -5
Episode 5780 | epsilon = 0.100 | reward = -8
Episode 5790 | epsilon = 0.100 | reward =

Episode 7410 | epsilon = 0.100 | reward = -5
Episode 7420 | epsilon = 0.100 | reward = -8
Episode 7430 | epsilon = 0.100 | reward = -14
Episode 7440 | epsilon = 0.100 | reward = -8
Episode 7450 | epsilon = 0.100 | reward = -8
Episode 7460 | epsilon = 0.100 | reward = -32
Episode 7470 | epsilon = 0.100 | reward = -11
Episode 7480 | epsilon = 0.100 | reward = -20
Episode 7490 | epsilon = 0.100 | reward = -41
Episode 7500 | epsilon = 0.100 | reward = -17
Episode 7510 | epsilon = 0.100 | reward = -29
Episode 7520 | epsilon = 0.100 | reward = -14
Episode 7530 | epsilon = 0.100 | reward = -11
Episode 7540 | epsilon = 0.100 | reward = -11
Episode 7550 | epsilon = 0.100 | reward = -14
Episode 7560 | epsilon = 0.100 | reward = -8
Episode 7570 | epsilon = 0.100 | reward = -29
Episode 7580 | epsilon = 0.100 | reward = -59
Episode 7590 | epsilon = 0.100 | reward = -17
Episode 7600 | epsilon = 0.100 | reward = -5
Episode 7610 | epsilon = 0.100 | reward = -14
Episode 7620 | epsilon = 0.100 | reward 

...

Episode 11150 | epsilon = 0.100 | reward = -7
Episode 11160 | epsilon = 0.100 | reward = -8
Episode 11170 | epsilon = 0.100 | reward = -23
Episode 11180 | epsilon = 0.100 | reward = -11
Episode 11190 | epsilon = 0.100 | reward = -5
Episode 11200 | epsilon = 0.100 | reward = -8
Episode 11210 | epsilon = 0.100 | reward = -53
Episode 11220 | epsilon = 0.100 | reward = -14
Episode 11230 | epsilon = 0.100 | reward = -8
Episode 11240 | epsilon = 0.100 | reward = -29
Episode 11250 | epsilon = 0.100 | reward = -5
Episode 11260 | epsilon = 0.100 | reward = -5
Episode 11270 | epsilon = 0.100 | reward = -8
Episode 11280 | epsilon = 0.100 | reward = -8
Episode 11290 | epsilon = 0.100 | reward = -5
Episode 11300 | epsilon = 0.100 | reward = -8
Episode 11310 | epsilon = 0.100 | reward = -13
Episode 11320 | epsilon = 0.100 | reward = -11
Episode 11330 | epsilon = 0.100 | reward = -8
Episode 11340 | epsilon = 0.100 | reward = -5
Episode 11350 | epsilon = 0.100 | reward = -32
Episode 11360 | epsilon = 

Episode 12930 | epsilon = 0.100 | reward = -5
Episode 12940 | epsilon = 0.100 | reward = -5
Episode 12950 | epsilon = 0.100 | reward = -8
Episode 12960 | epsilon = 0.100 | reward = -11
Episode 12970 | epsilon = 0.100 | reward = -8
Episode 12980 | epsilon = 0.100 | reward = -8
Episode 12990 | epsilon = 0.100 | reward = -26
Episode 13000 | epsilon = 0.100 | reward = -11
Episode 13010 | epsilon = 0.100 | reward = -20
Episode 13020 | epsilon = 0.100 | reward = -17
Episode 13030 | epsilon = 0.100 | reward = -14
Episode 13040 | epsilon = 0.100 | reward = -8
Episode 13050 | epsilon = 0.100 | reward = -8
Episode 13060 | epsilon = 0.100 | reward = -16
Episode 13070 | epsilon = 0.100 | reward = -5
Episode 13080 | epsilon = 0.100 | reward = -5
Episode 13090 | epsilon = 0.100 | reward = -68
Episode 13100 | epsilon = 0.100 | reward = -32
Episode 13110 | epsilon = 0.100 | reward = -5
Episode 13120 | epsilon = 0.100 | reward = -44
Episode 13130 | epsilon = 0.100 | reward = -11
Episode 13140 | epsilon

Episode 14760 | epsilon = 0.100 | reward = -23
Episode 14770 | epsilon = 0.100 | reward = -5
Episode 14780 | epsilon = 0.100 | reward = -28
Episode 14790 | epsilon = 0.100 | reward = -11
Episode 14800 | epsilon = 0.100 | reward = -8
Episode 14810 | epsilon = 0.100 | reward = -8
Episode 14820 | epsilon = 0.100 | reward = -11
Episode 14830 | epsilon = 0.100 | reward = -17
Episode 14840 | epsilon = 0.100 | reward = -29
Episode 14850 | epsilon = 0.100 | reward = -20
Episode 14860 | epsilon = 0.100 | reward = -20
Episode 14870 | epsilon = 0.100 | reward = -17
Episode 14880 | epsilon = 0.100 | reward = -17
Episode 14890 | epsilon = 0.100 | reward = -8
Episode 14900 | epsilon = 0.100 | reward = -14
Episode 14910 | epsilon = 0.100 | reward = -8
Episode 14920 | epsilon = 0.100 | reward = -11
Episode 14930 | epsilon = 0.100 | reward = -5
Episode 14940 | epsilon = 0.100 | reward = -17
Episode 14950 | epsilon = 0.100 | reward = -11
Episode 14960 | epsilon = 0.100 | reward = -23
Episode 14970 | eps

Episode 16540 | epsilon = 0.100 | reward = -77
Episode 16550 | epsilon = 0.100 | reward = -5
Episode 16560 | epsilon = 0.100 | reward = -7
Episode 16570 | epsilon = 0.100 | reward = -8
Episode 16580 | epsilon = 0.100 | reward = -14
Episode 16590 | epsilon = 0.100 | reward = -38
Episode 16600 | epsilon = 0.100 | reward = -8
Episode 16610 | epsilon = 0.100 | reward = -20
Episode 16620 | epsilon = 0.100 | reward = -17
Episode 16630 | epsilon = 0.100 | reward = -50
Episode 16640 | epsilon = 0.100 | reward = -5
Episode 16650 | epsilon = 0.100 | reward = -1
Episode 16660 | epsilon = 0.100 | reward = -26
Episode 16670 | epsilon = 0.100 | reward = -20
Episode 16680 | epsilon = 0.100 | reward = -8
Episode 16690 | epsilon = 0.100 | reward = -8
Episode 16700 | epsilon = 0.100 | reward = -8
Episode 16710 | epsilon = 0.100 | reward = -5
Episode 16720 | epsilon = 0.100 | reward = -5
Episode 16730 | epsilon = 0.100 | reward = -20
Episode 16740 | epsilon = 0.100 | reward = -4
Episode 16750 | epsilon =

Episode 18390 | epsilon = 0.100 | reward = -5
Episode 18400 | epsilon = 0.100 | reward = -17
Episode 18410 | epsilon = 0.100 | reward = -8
Episode 18420 | epsilon = 0.100 | reward = -11
Episode 18430 | epsilon = 0.100 | reward = -11
Episode 18440 | epsilon = 0.100 | reward = -22
Episode 18450 | epsilon = 0.100 | reward = -11
Episode 18460 | epsilon = 0.100 | reward = -11
Episode 18470 | epsilon = 0.100 | reward = -17
Episode 18480 | epsilon = 0.100 | reward = -5
Episode 18490 | epsilon = 0.100 | reward = -8
Episode 18500 | epsilon = 0.100 | reward = -10
Episode 18510 | epsilon = 0.100 | reward = -20
Episode 18520 | epsilon = 0.100 | reward = -5
Episode 18530 | epsilon = 0.100 | reward = -8
Episode 18540 | epsilon = 0.100 | reward = -5
Episode 18550 | epsilon = 0.100 | reward = -11
Episode 18560 | epsilon = 0.100 | reward = -5
Episode 18570 | epsilon = 0.100 | reward = -7
Episode 18580 | epsilon = 0.100 | reward = -17
Episode 18590 | epsilon = 0.100 | reward = -7
Episode 18600 | epsilon

In [20]:
display_tabular(qValues_qLearning_hardMap2, hard_map2)

Chosen action: 1
Chosen action: 2
Chosen action: 1
Chosen action: 0
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 0


### b- Sarsa

In [21]:
gamma = 0.9
learning_rate = 1
max_episodes = 20000

qValues_sarsa_hardMap2 = sarsa(hard_env2, gamma, learning_rate, max_episodes)
for row in qValues_sarsa_hardMap2:
    print(["{:.2f}".format(x) for x in row])

Episode 10 | epsilon = 0.914 | reward = -8
Episode 20 | epsilon = 0.826 | reward = -14
Episode 30 | epsilon = 0.747 | reward = -29
Episode 40 | epsilon = 0.676 | reward = -17
Episode 50 | epsilon = 0.611 | reward = -5
Episode 60 | epsilon = 0.553 | reward = -20
Episode 70 | epsilon = 0.500 | reward = -8
Episode 80 | epsilon = 0.452 | reward = -5
Episode 90 | epsilon = 0.409 | reward = -8
Episode 100 | epsilon = 0.370 | reward = -23
Episode 110 | epsilon = 0.334 | reward = -14
Episode 120 | epsilon = 0.302 | reward = -14
Episode 130 | epsilon = 0.273 | reward = -14
Episode 140 | epsilon = 0.247 | reward = -5
Episode 150 | epsilon = 0.224 | reward = -8
Episode 160 | epsilon = 0.202 | reward = -20
Episode 170 | epsilon = 0.183 | reward = -20
Episode 180 | epsilon = 0.165 | reward = -35
Episode 190 | epsilon = 0.150 | reward = 2
Episode 200 | epsilon = 0.135 | reward = -8
Episode 210 | epsilon = 0.122 | reward = -5
Episode 220 | epsilon = 0.111 | reward = -5
Episode 230 | epsilon = 0.100 |

Episode 1870 | epsilon = 0.100 | reward = -8
Episode 1880 | epsilon = 0.100 | reward = -8
Episode 1890 | epsilon = 0.100 | reward = -11
Episode 1900 | epsilon = 0.100 | reward = -8
Episode 1910 | epsilon = 0.100 | reward = -8
Episode 1920 | epsilon = 0.100 | reward = -5
Episode 1930 | epsilon = 0.100 | reward = -11
Episode 1940 | epsilon = 0.100 | reward = -17
Episode 1950 | epsilon = 0.100 | reward = -8
Episode 1960 | epsilon = 0.100 | reward = -11
Episode 1970 | epsilon = 0.100 | reward = 2
Episode 1980 | epsilon = 0.100 | reward = -1
Episode 1990 | epsilon = 0.100 | reward = -5
Episode 2000 | epsilon = 0.100 | reward = -11
Episode 2010 | epsilon = 0.100 | reward = -8
Episode 2020 | epsilon = 0.100 | reward = -8
Episode 2030 | epsilon = 0.100 | reward = -5
Episode 2040 | epsilon = 0.100 | reward = -17
Episode 2050 | epsilon = 0.100 | reward = -5
Episode 2060 | epsilon = 0.100 | reward = -18
Episode 2070 | epsilon = 0.100 | reward = -10
Episode 2080 | epsilon = 0.100 | reward = -5
Epi

Episode 3970 | epsilon = 0.100 | reward = -7
Episode 3980 | epsilon = 0.100 | reward = -11
Episode 3990 | epsilon = 0.100 | reward = -28
Episode 4000 | epsilon = 0.100 | reward = -1
Episode 4010 | epsilon = 0.100 | reward = -14
Episode 4020 | epsilon = 0.100 | reward = -32
Episode 4030 | epsilon = 0.100 | reward = -8
Episode 4040 | epsilon = 0.100 | reward = -5
Episode 4050 | epsilon = 0.100 | reward = -11
Episode 4060 | epsilon = 0.100 | reward = 5
Episode 4070 | epsilon = 0.100 | reward = 2
Episode 4080 | epsilon = 0.100 | reward = 5
Episode 4090 | epsilon = 0.100 | reward = -8
Episode 4100 | epsilon = 0.100 | reward = -20
Episode 4110 | epsilon = 0.100 | reward = -5
Episode 4120 | epsilon = 0.100 | reward = -5
Episode 4130 | epsilon = 0.100 | reward = -17
Episode 4140 | epsilon = 0.100 | reward = -14
Episode 4150 | epsilon = 0.100 | reward = -1
Episode 4160 | epsilon = 0.100 | reward = -5
Episode 4170 | epsilon = 0.100 | reward = 5
Episode 4180 | epsilon = 0.100 | reward = -5
Episod

Episode 5930 | epsilon = 0.100 | reward = -5
Episode 5940 | epsilon = 0.100 | reward = -7
Episode 5950 | epsilon = 0.100 | reward = -5
Episode 5960 | epsilon = 0.100 | reward = -17
Episode 5970 | epsilon = 0.100 | reward = -4
Episode 5980 | epsilon = 0.100 | reward = -1
Episode 5990 | epsilon = 0.100 | reward = -5
Episode 6000 | epsilon = 0.100 | reward = -11
Episode 6010 | epsilon = 0.100 | reward = -7
Episode 6020 | epsilon = 0.100 | reward = -5
Episode 6030 | epsilon = 0.100 | reward = -11
Episode 6040 | epsilon = 0.100 | reward = 5
Episode 6050 | epsilon = 0.100 | reward = -5
Episode 6060 | epsilon = 0.100 | reward = -5
Episode 6070 | epsilon = 0.100 | reward = -14
Episode 6080 | epsilon = 0.100 | reward = -14
Episode 6090 | epsilon = 0.100 | reward = -14
Episode 6100 | epsilon = 0.100 | reward = -8
Episode 6110 | epsilon = 0.100 | reward = -8
Episode 6120 | epsilon = 0.100 | reward = -11
Episode 6130 | epsilon = 0.100 | reward = -11
Episode 6140 | epsilon = 0.100 | reward = -5
Epi

Episode 7960 | epsilon = 0.100 | reward = -11
Episode 7970 | epsilon = 0.100 | reward = -5
Episode 7980 | epsilon = 0.100 | reward = -5
Episode 7990 | epsilon = 0.100 | reward = -14
Episode 8000 | epsilon = 0.100 | reward = -8
Episode 8010 | epsilon = 0.100 | reward = -8
Episode 8020 | epsilon = 0.100 | reward = -5
Episode 8030 | epsilon = 0.100 | reward = -8
Episode 8040 | epsilon = 0.100 | reward = -8
Episode 8050 | epsilon = 0.100 | reward = -14
Episode 8060 | epsilon = 0.100 | reward = -8
Episode 8070 | epsilon = 0.100 | reward = -8
Episode 8080 | epsilon = 0.100 | reward = -11
Episode 8090 | epsilon = 0.100 | reward = -14
Episode 8100 | epsilon = 0.100 | reward = -5
Episode 8110 | epsilon = 0.100 | reward = -8
Episode 8120 | epsilon = 0.100 | reward = -14
Episode 8130 | epsilon = 0.100 | reward = -20
Episode 8140 | epsilon = 0.100 | reward = -32
Episode 8150 | epsilon = 0.100 | reward = -8
Episode 8160 | epsilon = 0.100 | reward = -14
Episode 8170 | epsilon = 0.100 | reward = -11


...

Episode 11700 | epsilon = 0.100 | reward = -5
Episode 11710 | epsilon = 0.100 | reward = -8
Episode 11720 | epsilon = 0.100 | reward = -5
Episode 11730 | epsilon = 0.100 | reward = -8
Episode 11740 | epsilon = 0.100 | reward = -5
Episode 11750 | epsilon = 0.100 | reward = -11
Episode 11760 | epsilon = 0.100 | reward = -23
Episode 11770 | epsilon = 0.100 | reward = -5
Episode 11780 | epsilon = 0.100 | reward = -8
Episode 11790 | epsilon = 0.100 | reward = -7
Episode 11800 | epsilon = 0.100 | reward = -14
Episode 11810 | epsilon = 0.100 | reward = -14
Episode 11820 | epsilon = 0.100 | reward = -17
Episode 11830 | epsilon = 0.100 | reward = -14
Episode 11840 | epsilon = 0.100 | reward = -5
Episode 11850 | epsilon = 0.100 | reward = -11
Episode 11860 | epsilon = 0.100 | reward = -20
Episode 11870 | epsilon = 0.100 | reward = -11
Episode 11880 | epsilon = 0.100 | reward = -14
Episode 11890 | epsilon = 0.100 | reward = -5
Episode 11900 | epsilon = 0.100 | reward = -5
Episode 11910 | epsilon 

Episode 13570 | epsilon = 0.100 | reward = -8
Episode 13580 | epsilon = 0.100 | reward = 2
Episode 13590 | epsilon = 0.100 | reward = -8
Episode 13600 | epsilon = 0.100 | reward = -5
Episode 13610 | epsilon = 0.100 | reward = -5
Episode 13620 | epsilon = 0.100 | reward = -8
Episode 13630 | epsilon = 0.100 | reward = -5
Episode 13640 | epsilon = 0.100 | reward = -17
Episode 13650 | epsilon = 0.100 | reward = -8
Episode 13660 | epsilon = 0.100 | reward = -11
Episode 13670 | epsilon = 0.100 | reward = 5
Episode 13680 | epsilon = 0.100 | reward = -11
Episode 13690 | epsilon = 0.100 | reward = -8
Episode 13700 | epsilon = 0.100 | reward = -5
Episode 13710 | epsilon = 0.100 | reward = -11
Episode 13720 | epsilon = 0.100 | reward = -5
Episode 13730 | epsilon = 0.100 | reward = -11
Episode 13740 | epsilon = 0.100 | reward = -8
Episode 13750 | epsilon = 0.100 | reward = -5
Episode 13760 | epsilon = 0.100 | reward = -8
Episode 13770 | epsilon = 0.100 | reward = -8
Episode 13780 | epsilon = 0.100

Episode 15420 | epsilon = 0.100 | reward = -8
Episode 15430 | epsilon = 0.100 | reward = -8
Episode 15440 | epsilon = 0.100 | reward = -1
Episode 15450 | epsilon = 0.100 | reward = -23
Episode 15460 | epsilon = 0.100 | reward = -47
Episode 15470 | epsilon = 0.100 | reward = -5
Episode 15480 | epsilon = 0.100 | reward = -8
Episode 15490 | epsilon = 0.100 | reward = -5
Episode 15500 | epsilon = 0.100 | reward = -11
Episode 15510 | epsilon = 0.100 | reward = -10
Episode 15520 | epsilon = 0.100 | reward = 2
Episode 15530 | epsilon = 0.100 | reward = -26
Episode 15540 | epsilon = 0.100 | reward = -5
Episode 15550 | epsilon = 0.100 | reward = -17
Episode 15560 | epsilon = 0.100 | reward = -44
Episode 15570 | epsilon = 0.100 | reward = -11
Episode 15580 | epsilon = 0.100 | reward = -1
Episode 15590 | epsilon = 0.100 | reward = -8
Episode 15600 | epsilon = 0.100 | reward = -17
Episode 15610 | epsilon = 0.100 | reward = -5
Episode 15620 | epsilon = 0.100 | reward = -11
Episode 15630 | epsilon =

Episode 17570 | epsilon = 0.100 | reward = -7
Episode 17580 | epsilon = 0.100 | reward = 2
Episode 17590 | epsilon = 0.100 | reward = -11
Episode 17600 | epsilon = 0.100 | reward = -10
Episode 17610 | epsilon = 0.100 | reward = -14
Episode 17620 | epsilon = 0.100 | reward = -5
Episode 17630 | epsilon = 0.100 | reward = -11
Episode 17640 | epsilon = 0.100 | reward = -8
Episode 17650 | epsilon = 0.100 | reward = -8
Episode 17660 | epsilon = 0.100 | reward = -20
Episode 17670 | epsilon = 0.100 | reward = 5
Episode 17680 | epsilon = 0.100 | reward = -14
Episode 17690 | epsilon = 0.100 | reward = -5
Episode 17700 | epsilon = 0.100 | reward = -23
Episode 17710 | epsilon = 0.100 | reward = -16
Episode 17720 | epsilon = 0.100 | reward = -14
Episode 17730 | epsilon = 0.100 | reward = -5
Episode 17740 | epsilon = 0.100 | reward = -1
Episode 17750 | epsilon = 0.100 | reward = -8
Episode 17760 | epsilon = 0.100 | reward = -17
Episode 17770 | epsilon = 0.100 | reward = -14
Episode 17780 | epsilon =

Episode 19580 | epsilon = 0.100 | reward = -11
Episode 19590 | epsilon = 0.100 | reward = -20
Episode 19600 | epsilon = 0.100 | reward = -8
Episode 19610 | epsilon = 0.100 | reward = -14
Episode 19620 | epsilon = 0.100 | reward = -17
Episode 19630 | epsilon = 0.100 | reward = -8
Episode 19640 | epsilon = 0.100 | reward = -5
Episode 19650 | epsilon = 0.100 | reward = 5
Episode 19660 | epsilon = 0.100 | reward = 2
Episode 19670 | epsilon = 0.100 | reward = -5
Episode 19680 | epsilon = 0.100 | reward = 2
Episode 19690 | epsilon = 0.100 | reward = -5
Episode 19700 | epsilon = 0.100 | reward = -8
Episode 19710 | epsilon = 0.100 | reward = -5
Episode 19720 | epsilon = 0.100 | reward = -5
Episode 19730 | epsilon = 0.100 | reward = -4
Episode 19740 | epsilon = 0.100 | reward = -14
Episode 19750 | epsilon = 0.100 | reward = -11
Episode 19760 | epsilon = 0.100 | reward = -5
Episode 19770 | epsilon = 0.100 | reward = -5
Episode 19780 | epsilon = 0.100 | reward = -7
Episode 19790 | epsilon = 0.100

In [22]:
display_tabular(qValues_sarsa_hardMap2, hard_map2)

Chosen action: 2
Chosen action: 1
Chosen action: 2
Chosen action: 2
Chosen action: 2
Chosen action: 0
Chosen action: 2
Chosen action: 0
Chosen action: 0
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 1
Chosen action: 2
Chosen action: 1
Chosen action: 1
Chosen action: 0
Chosen action: 3
Chosen action: 3
Chosen action: 3
Chosen action: 1
Chosen action: 1
Chosen action: 3
Chosen action: 1
Chosen action: 1
Chosen action: 2
Chosen action: 2
Chosen action: 0
Chosen action: 2
Chosen action: 2
Chosen action: 1
Chosen action: 1
Chosen action: 0
Chosen action: 3
Chosen action: 3
Chosen action: 1
Chosen action: 0
Chosen action: 3
Chosen action: 3
Chosen action: 0


# Evaluation des résultats

In [23]:
# Cette fonction permet de determiner si l'agent se trouve dans une situation dangereuse ou non
def is_dangerous(desc, row, col):

    nrow = len(desc)
    ncol = len(desc[0])

    directions = [
        (-1, 0),  # haut
        (1, 0),   # bas
        (0, -1),  # gauche
        (0, 1)    # droite
    ]

    for dr, dc in directions:
        r, c = row + dr, col + dc

        if 0 <= r < nrow and 0 <= c < ncol:
            if desc[r][c].decode("utf-8") == 'H':
                return True

    return False

In [24]:
# Cette fonction permet de collecter un ensemble de métriques pour évaluer la performance de l'agent sur un ensemble d'épisodes
# TODO : Créer des figures pour visualiser la performance de l'agent par rapport à ces métriques
# TODO : Chercher autres métriques pertinentes
def evaluate_agent(env, Q, n_episodes=1000, max_steps=100):

    ncol = env.unwrapped.ncol
    desc = env.unwrapped.desc

    success = 0
    falls = 0
    wall_hits = 0
    danger_states = 0
    total_steps = 0
    episodes_steps = []
    episodes_wall_hits = []
    episodes_danger_states = []

    for episode in range(n_episodes):

        state, _ = env.reset()
        done = False
        episode_step = 0
        episode_wall_hits = 0
        episode_danger_states = 0

        for step in range(max_steps):
            total_steps +=1
            episode_step += 1
            
            action = numpy.argmax(Q[state])

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # coordonnées
            row, col = divmod(next_state, ncol)
            tile = desc[row][col].decode("utf-8")

            if tile == 'G':
                success += 1

            elif tile == 'H':
                falls += 1

            if next_state == state:
                wall_hits += 1
                episode_wall_hits +=1

            if is_dangerous(desc, row, col):
                danger_states += 1
                episode_danger_states += 1

            state = next_state

            if done:
                break
        episodes_steps.append(episode_step)
        episodes_wall_hits.append(episode_wall_hits)
        episodes_danger_states.append(episode_danger_states)

    avg_steps = sum(episodes_steps) / n_episodes
    avg_wall_hits = sum(episodes_wall_hits) / n_episodes
    avg_danger_states = sum(episodes_danger_states) / n_episodes
    loops = n_episodes - success - falls
    wall_hits_per_cent = wall_hits / total_steps
    danger_states_per_cent = danger_states / total_steps
    
    return {
        "average steps": avg_steps,
        "success": success,
        "falls": falls,
        "loops": loops,
        "wall_hits": wall_hits,
        "wall_hits%": wall_hits_per_cent,
        "average wall hits per episode": avg_wall_hits,
        "danger_states": danger_states,
        "danger_states%": danger_states_per_cent,
        "average danger states per episode": avg_danger_states
    }

### a- Environnement simple

#### i- Q-learning

In [25]:
evaluate_agent(safe_env, qValues_qLearning_safeMap)

{'average steps': 59.26,
 'success': 546,
 'falls': 188,
 'loops': 266,
 'wall_hits': 13973,
 'wall_hits%': 0.2357914276071549,
 'average wall hits per episode': 13.973,
 'danger_states': 0.0508268646641917,
 'average danger states per episode': 3.012}

#### ii- Sarsa

In [26]:
evaluate_agent(safe_env, qValues_sarsa_safeMap)

{'average steps': 15.71,
 'success': 730,
 'falls': 270,
 'loops': 0,
 'wall_hits': 1634,
 'wall_hits%': 0.10401018459579886,
 'average wall hits per episode': 1.634,
 'danger_states': 0.24061107574793125,
 'average danger states per episode': 3.78}

### b- Premier envirronement difficile

#### i- Q-learning

In [27]:
evaluate_agent(hard_env1, qValues_qLearning_hardMap1)

{'average steps': 49.266,
 'success': 510,
 'falls': 330,
 'loops': 160,
 'wall_hits': 3476,
 'wall_hits%': 0.07055575853529818,
 'average wall hits per episode': 3.476,
 'danger_states': 0.1754353915479235,
 'average danger states per episode': 8.643}

#### ii- Sarsa

In [28]:
evaluate_agent(hard_env1, qValues_sarsa_hardMap1)

{'average steps': 31.426,
 'success': 601,
 'falls': 377,
 'loops': 22,
 'wall_hits': 1534,
 'wall_hits%': 0.04881308470693057,
 'average wall hits per episode': 1.534,
 'danger_states': 0.25981671227645897,
 'average danger states per episode': 8.165}

### Deuxième environnement difficile

#### i- Q-learning

In [29]:
evaluate_agent(hard_env2, qValues_qLearning_hardMap2)

{'average steps': 23.34,
 'success': 260,
 'falls': 739,
 'loops': 1,
 'wall_hits': 3353,
 'wall_hits%': 0.14365895458440445,
 'average wall hits per episode': 3.353,
 'danger_states': 0.412853470437018,
 'average danger states per episode': 9.636}

#### ii- Sarsa

In [30]:
evaluate_agent(hard_env2, qValues_sarsa_hardMap2)

{'average steps': 23.443,
 'success': 238,
 'falls': 754,
 'loops': 8,
 'wall_hits': 1314,
 'wall_hits%': 0.05605084673463294,
 'average wall hits per episode': 1.314,
 'danger_states': 0.39308109030414196,
 'average danger states per episode': 9.215}